In [2]:
import torch
import torch.nn.functional as F

x = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])

print(x.shape)
print(x)

torch.Size([3, 2])
tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])


In [5]:
print(x.T.shape)
print(x.T)

torch.Size([2, 3])
tensor([[1., 0., 1.],
        [0., 1., 1.]])


In [6]:
scores = x @ x.T
print(scores.shape)
print(scores)

torch.Size([3, 3])
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])


In [7]:
weights = F.softmax(scores, dim=-1)
print(weights)

tensor([[0.4223, 0.1554, 0.4223],
        [0.1554, 0.4223, 0.4223],
        [0.2119, 0.2119, 0.5761]])


In [8]:
out = weights @ x
print(out.shape)
print(out)

torch.Size([3, 2])
tensor([[0.8446, 0.5777],
        [0.5777, 0.8446],
        [0.7881, 0.7881]])


In [9]:
import torch.nn as nn

torch.manual_seed(0)

W_q = nn.Linear(2, 2, bias=False)
print(W_q.weight)

q = W_q(x)
print(q.shape)
print(q)

Parameter containing:
tensor([[-0.0053,  0.3793],
        [-0.5820, -0.5204]], requires_grad=True)
torch.Size([3, 2])
tensor([[-0.0053, -0.5820],
        [ 0.3793, -0.5204],
        [ 0.3740, -1.1024]], grad_fn=<MmBackward0>)


In [10]:
W_k = nn.Linear(2, 2, bias=False)
W_v = nn.Linear(2, 2, bias=False)

k = W_k(x)
v = W_v(x)

scores = q @ k.T
print(scores)

tensor([[ 0.0096, -0.3273, -0.3177],
        [-0.0960, -0.2198, -0.3158],
        [-0.0864, -0.5471, -0.6336]], grad_fn=<MmBackward0>)


In [11]:
d = k.shape[-1]

weights = F.softmax(scores / d ** 0.5, dim=-1)
print(weights)

out = weights @ v
print(out.shape)
print(out)

tensor([[0.3874, 0.3053, 0.3073],
        [0.3607, 0.3305, 0.3088],
        [0.4165, 0.3007, 0.2829]], grad_fn=<SoftmaxBackward0>)
torch.Size([3, 2])
tensor([[ 0.0710, -0.2336],
        [ 0.0776, -0.2319],
        [ 0.0653, -0.2305]], grad_fn=<MmBackward0>)


In [12]:
mask = torch.tril(torch.ones(3, 3))
print(mask)

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


In [13]:
scores_masked = scores.masked_fill(mask == 0, float("-inf"))
print(scores_masked)

weights = F.softmax(scores_masked / d ** 0.5, dim=-1)
print(weights)

out = weights @ v
print(out)

tensor([[ 0.0096,    -inf,    -inf],
        [-0.0960, -0.2198,    -inf],
        [-0.0864, -0.5471, -0.6336]], grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000],
        [0.5219, 0.4781, 0.0000],
        [0.4165, 0.3007, 0.2829]], grad_fn=<SoftmaxBackward0>)
tensor([[-0.0628, -0.2137],
        [ 0.0567, -0.1780],
        [ 0.0653, -0.2305]], grad_fn=<MmBackward0>)


In [17]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim, head_dim):
        super().__init__()
        self.W_q = nn.Linear(embedding_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embedding_dim, head_dim, bias=False)
        self.W_v = nn.Linear(embedding_dim, head_dim, bias=False)

    def forward(self, x):
        seq_len = x.shape[-2]

        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        scores = q @ k.transpose(-2, -1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))

        weights = F.softmax(scores / k.shape[-1] ** 0.5, dim=-1)
        return weights @ v

In [21]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        head_dim = embedding_dim // num_heads
        self.heads = nn.ModuleList(
            [CausalSelfAttention(embedding_dim, head_dim) for _ in range(num_heads)]
        )
        self.W_o = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.W_o(out)


torch.manual_seed(0)
mha = MultiHeadAttention(embedding_dim=8, num_heads=4)

batch = torch.randn(2, 5, 8)

print(mha.heads[0](batch).shape)
print(mha(batch).shape)

torch.Size([2, 5, 2])
torch.Size([2, 5, 8])
